In [1]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(model = "openai/gpt-oss-120b", 
               api_key= os.environ["GROQ_API_COLD_EMAIL"],
               temperature=0)

response = llm.invoke("How far is moon from Earth?")

print(response.content)

The Moon orbits Earth at an average distance of **about 384 400 kilometers (≈ 238 900 miles)**.

Because the Moon’s orbit is slightly elliptical, the actual distance varies over the course of a month:

| Orbital point | Distance from Earth | Approx. light‑travel time |
|---------------|---------------------|---------------------------|
| **Perigee** (closest) | ~ 363 300 km (≈ 225 700 mi) | ~ 1.21 seconds |
| **Apogee** (farthest) | ~ 405 500 km (≈ 251 900 mi) | ~ 1.35 seconds |
| **Mean (average)** | ~ 384 400 km (≈ 238 900 mi) | ~ 1.28 seconds |

So, on a typical night the Moon is roughly 384 000 km away, but it can be a few tens of thousands of kilometers nearer or farther depending on where it is in its elliptical orbit.


In [2]:
# WebBase loader Class from LangChain to load the content of a webpage and return it as a string. It uses the requests library to fetch the webpage and BeautifulSoup to parse the HTML content. The class has a method called load that takes a URL as input and returns the text content of the webpage.

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.siemens-energy.com/en_US/Jobs/FolderDetail?folderId=298438&source=LinkedIn")

page_data = loader.load().pop().page_content
print(page_data)


/tmp/ipykernel_16047/1355990012.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.















        AI/Machine Learning Engineer - India - 298438 - Siemens Energy - Careers Marketplace
    
















                        Skip to content                    

                        Skip to footer                    




                                    Siemens Energy
                                




            Menu        






                        Search Jobs                    



                        About us                    



                        Early careers                    



                        Hiring process                    





                                Login                            







                                Current language
                            


DeutschEnglishEspañolFrançaisMagyarNederlandsPortuguêsRomânăČeština简体中文繁體中文日本語














        Siemens Energy
    

Siemens Energy and Siemens Gamesa are becoming Omterra. "We energize society" by supporting our customers to make t

In [14]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate(
    input_variables=["page_data"],
    template="""
You are a job description extraction assistant.

Extract the job details from the following webpage content.

Return the result as valid JSON with exactly these four keys:
- role: Job title
- experience_needed: Required years of experience, preserving the stated range, or "Not specified"
- skills: Array of technical and soft skills explicitly mentioned
- description: Concise summary of responsibilities, duties, and expectations

Instructions:
1. Use only information available in the provided webpage content.
2. Do not invent or assume missing information.
3. Ignore unrelated content, advertisements, navigation menus, and recommended jobs.
4. If a field is missing, use "Not specified" for string fields and [] for skills.
5. Return ONLY valid JSON. Do not include preamble, explanations, or Markdown code fences.

Webpage content:
{page_data}

Required JSON format:
{{
  "role": "Job title",
  "company": "Company name",
  "experience_needed": "Required experience",
  "skills": ["Skill 1", "Skill 2"],
  "description": "Job description summary"
}}
"""
)

chain_extract = prompt_extract | llm

response = chain_extract.invoke({"page_data": page_data})

print(response.content)
type(response.content)

{
  "role": "AI/Machine Learning Engineer",
  "company": "Siemens Energy Industrial Turbomachinery India Private Limited",
  "experience_needed": "1+ years",
  "skills": [
    "Python",
    "Problem solving",
    "Machine Learning concepts",
    "Natural Language Processing (NLP)",
    "Model evaluation",
    "NumPy",
    "Pandas",
    "scikit-learn",
    "PyTorch",
    "TensorFlow",
    "Generative AI concepts",
    "Large Language Models (LLMs)",
    "Prompt engineering",
    "Retrieval Augmented Generation (RAG)",
    "API development",
    "FastAPI",
    "Backend development",
    "AWS",
    "Azure",
    "Git",
    "Software development best practices",
    "Data preprocessing",
    "Text cleaning",
    "Parsing",
    "Chunking",
    "Embeddings",
    "Collaboration",
    "Adaptability"
  ],
  "description": "The AI/Machine Learning Engineer will design, develop, and maintain machine learning and generative AI pipelines, collaborate with cross‑functional teams, build backend APIs (

str

In [4]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_response = json_parser.parse(response.content)

json_response
print(type(json_response))
json_response

<class 'dict'>


{'role': 'AI/Machine Learning Engineer',
 'experience_needed': '1+ years',
 'skills': ['Python',
  'Problem-solving',
  'Machine Learning concepts',
  'Natural Language Processing (NLP)',
  'Model evaluation',
  'NumPy',
  'Pandas',
  'scikit-learn',
  'PyTorch',
  'TensorFlow',
  'Generative AI (LLMs, prompt engineering, RAG)',
  'FastAPI',
  'API development',
  'AWS',
  'Azure',
  'Git',
  'Software development best practices',
  'Willingness to learn',
  'Adaptability'],
 'description': 'Design, develop, and maintain machine learning and generative AI pipelines, including data preprocessing, model training, evaluation, and inference. Build NLP workflows, process text data, and create backend APIs (e.g., FastAPI) to expose AI functionalities. Integrate AI models into applications, write clean modular Python code, and collaborate with data scientists, software engineers, and product managers. Debug, test, and optimize models for performance and reliability while staying current with 

In [5]:
import pandas as pd

df = pd.read_csv("/workspaces/Cold-Email-Generator/Portfolio-Likitha - Sheet1.csv")

df 

,Tech stack,Links
0,Power BI,https://github.com/Likitha10/Patient-Wait-List...
1,Python; OpenCV,https://github.com/Likitha10/Document-Scanner
2,SQL; Data Analysis,https://github.com/Likitha10/Netflix-Data-Anal...
3,Python; Machine Learning; Fraud Detection,https://github.com/Likitha10/TCR-Projects/tree...
4,Astronomy Research; Spectroscopy; Interstellar...,https://likitha10.github.io/LikithaMadhav.gith...


In [ ]:
import chromadb
import uuid
#Persistent client is used to store the vectorstore on disk so that it can be reused later. The collection is created if it does not exist, and the data from the dataframe is added to the collection if it is empty. Each document is assigned a unique ID using uuid4.
client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name = "portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(
            documents= row["Tech stack"],
            metadatas = {"links": row["Links"]},
            ids = [str(uuid.uuid4())])

In [15]:
links = collection.query(
    query_texts = job["skills"],
    n_results = 2).get('metadatas' , [])

links

[[{'links': 'https://github.com/Likitha10/Document-Scanner'},
  {'links': 'https://github.com/Likitha10/TCR-Projects/tree/main/Credit%20Card%20Fraud%20Detection'}],
 [{'links': 'https://github.com/Likitha10/Netflix-Data-Analysis'},
  {'links': 'https://github.com/Likitha10/TCR-Projects/tree/main/Credit%20Card%20Fraud%20Detection'}],
 [{'links': 'https://github.com/Likitha10/TCR-Projects/tree/main/Credit%20Card%20Fraud%20Detection'},
  {'links': 'https://github.com/Likitha10/Netflix-Data-Analysis'}],
 [{'links': 'https://github.com/Likitha10/Netflix-Data-Analysis'},
  {'links': 'https://github.com/Likitha10/TCR-Projects/tree/main/Credit%20Card%20Fraud%20Detection'}],
 [{'links': 'https://github.com/Likitha10/Netflix-Data-Analysis'},
  {'links': 'https://github.com/Likitha10/Patient-Wait-List-Data'}],
 [{'links': 'https://github.com/Likitha10/Document-Scanner'},
  {'links': 'https://likitha10.github.io/LikithaMadhav.github.io/documents/Constraining_the_density_of_HBC__in_the_ISM_and_DIB_

In [16]:
job = json_response
job["skills"]

['Python',
 'Problem-solving',
 'Machine Learning concepts',
 'Natural Language Processing (NLP)',
 'Model evaluation',
 'NumPy',
 'Pandas',
 'scikit-learn',
 'PyTorch',
 'TensorFlow',
 'Generative AI (LLMs, prompt engineering, RAG)',
 'FastAPI',
 'API development',
 'AWS',
 'Azure',
 'Git',
 'Software development best practices',
 'Willingness to learn',
 'Adaptability']

In [29]:
prompt_email = PromptTemplate.from_template(

"""
You are an expert cold-email writer. Write a short, personalized cold email on my behalf as Likitha Madhav M R, applying for the role provided below. My email is likithamadhav10@gmail.com.

My background: I have an M.S. in Astrophysics from KU Leuven, Belgium, and experience in Python, SQL, machine learning, data analysis, research pipelines, and LLM evaluation at Innodata. My projects include credit card fraud detection, PyTorch image classification, and SQL-based Netflix data analysis.

Inputs:
- Portfolio links: {link_list}
- Job description: {job_description}

Instructions:
1. Research the company using web resources if available. Identify its products, recent initiatives, or potential business challenges.
2. Review my portfolio links and select the most relevant project.
3. Identify one specific opportunity where my existing skills could help the company, and briefly propose a practical solution.
4. Write a natural, professional cold email that connects the company, the opportunity, my relevant experience, and the advertised role.
5. Include an engaging subject line, a clear call to action, and the most relevant portfolio link.
6. Keep the email under 150 words. Avoid generic statements, excessive flattery, and unnecessary details.
7. Never invent company facts, project results, or experience.
8. Use the provided inputs directly. **Never ask me to provide them again.**

Output only the subject line and final email, ready to send. No PREAMBLE, explanations, or Markdown formatting. Ensure the email is concise, professional, and personalized.
"""
)

In [22]:
job 

{'role': 'AI/Machine Learning Engineer',
 'experience_needed': '1+ years',
 'skills': ['Python',
  'Problem-solving',
  'Machine Learning concepts',
  'Natural Language Processing (NLP)',
  'Model evaluation',
  'NumPy',
  'Pandas',
  'scikit-learn',
  'PyTorch',
  'TensorFlow',
  'Generative AI (LLMs, prompt engineering, RAG)',
  'FastAPI',
  'API development',
  'AWS',
  'Azure',
  'Git',
  'Software development best practices',
  'Willingness to learn',
  'Adaptability'],
 'description': 'Design, develop, and maintain machine learning and generative AI pipelines, including data preprocessing, model training, evaluation, and inference. Build NLP workflows, process text data, and create backend APIs (e.g., FastAPI) to expose AI functionalities. Integrate AI models into applications, write clean modular Python code, and collaborate with data scientists, software engineers, and product managers. Debug, test, and optimize models for performance and reliability while staying current with 

In [30]:
chain_email = prompt_email | llm
res = chain_email.invoke({"link_list": links, "job_description": str(job)})  
print(res.content)


Boosting Your AI Pipelines with Proven ML & FastAPI Expertise

Hi Hiring Team,

I’m Likitha Madhav, an M.S. Astrophysics graduate with 1+ yr of ML engineering at Innodata, where I built end‑to‑end LLM evaluation pipelines and productionized models via FastAPI. I noticed your team is expanding generative‑AI and NLP services, and I see an opportunity to accelerate model deployment and monitoring for your upcoming text‑analytics product.

In my recent credit‑card‑fraud‑detection project (https://github.com/Likitha10/TCR-Projects/tree/main/Credit%20Card%20Fraud%20Detection) I combined Python, scikit‑learn, PyTorch, and FastAPI to deliver a real‑time inference API with automated data preprocessing and drift detection—exactly the stack you list.

Could we schedule a 15‑minute call to discuss how I can help streamline your AI pipelines?

Best regards,
Likitha Madhav M R
likithamadhav10@gmail.com
